# MUSINSA → FEEDIT Human-in-the-Loop Mapping

이 노트북의 목적은 **무신사 RAW JSON을 사람이 직접 확인하면서 브랜드/카테고리 매핑표를 만드는 것**입니다.

흐름:

1. RAW JSON 로드
2. 카테고리 depth/code/name 유니크 추출
3. 브랜드 code/name 유니크 추출
4. 등장 상품 수와 예시 상품 확인
5. 사람이 `review_status`, `feedit_*` 컬럼을 직접 채움
6. 미매핑/중복/오류 검증
7. 최종 매핑 CSV/JSON 저장

> 이 단계에서는 자동 fuzzy merge를 하지 않습니다. 잘못된 브랜드/카테고리 병합을 피하기 위해 사람이 최종 결정합니다.


In [ ]:
from pathlib import Path
import json
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

# JSON 파일 경로를 필요하면 수정하세요.
JSON_PATH = Path("20260902T061616.json")

# 결과 저장 폴더
OUTPUT_DIR = Path("mapping_review")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("JSON_PATH   :", JSON_PATH.resolve())
print("OUTPUT_DIR  :", OUTPUT_DIR.resolve())


In [ ]:
if not JSON_PATH.exists():
    raise FileNotFoundError(
        f"RAW JSON 파일을 찾을 수 없습니다: {JSON_PATH.resolve()}\n"
        "JSON_PATH를 실제 파일 경로로 수정하세요."
    )

with JSON_PATH.open("r", encoding="utf-8") as f:
    raw = json.load(f)

products = raw.get("products") or []
ranking = raw.get("ranking") or {}

print("ranking period :", ranking.get("period"))
print("gender         :", ranking.get("gender"))
print("category_code  :", ranking.get("category_code"))
print("products       :", len(products))
print("errors         :", len(raw.get("errors") or []))


## 1. 카테고리 유니크 추출

In [ ]:
category_rows = []

for item in products:
    product = item.get("product") or {}
    category = product.get("category") or {}
    goods_no = product.get("goods_no")
    product_name = product.get("name")

    for depth in range(1, 5):
        code = category.get(f"depth{depth}_code")
        name = category.get(f"depth{depth}_name")

        if code or name:
            category_rows.append({
                "depth": depth,
                "source_category_code": str(code) if code is not None else None,
                "source_category_name": name,
                "goods_no": goods_no,
                "product_name": product_name,
            })

category_raw_df = pd.DataFrame(category_rows)

category_summary_df = (
    category_raw_df
    .groupby(
        ["depth", "source_category_code", "source_category_name"],
        dropna=False,
    )
    .agg(
        product_count=("goods_no", "nunique"),
        example_goods_no=("goods_no", "first"),
        example_product=("product_name", "first"),
    )
    .reset_index()
    .sort_values(["depth", "source_category_code"], na_position="last")
    .reset_index(drop=True)
)

print("유니크 카테고리 수:", len(category_summary_df))
display(category_summary_df)


### 가장 깊은 카테고리만 보기

실제 상품 매핑 키는 보통 `depth4 → depth3 → depth2 → depth1` 순서로 가장 깊은 값을 사용합니다.


In [ ]:
deepest_rows = []

for item in products:
    product = item.get("product") or {}
    category = product.get("category") or {}

    selected = None

    for depth in (4, 3, 2, 1):
        code = category.get(f"depth{depth}_code")
        name = category.get(f"depth{depth}_name")

        if code or name:
            selected = {
                "depth": depth,
                "source_category_code": str(code) if code is not None else None,
                "source_category_name": name,
                "goods_no": product.get("goods_no"),
                "product_name": product.get("name"),
            }
            break

    if selected:
        deepest_rows.append(selected)

deepest_df = pd.DataFrame(deepest_rows)

deepest_summary_df = (
    deepest_df
    .groupby(
        ["depth", "source_category_code", "source_category_name"],
        dropna=False,
    )
    .agg(
        product_count=("goods_no", "nunique"),
        example_goods_no=("goods_no", "first"),
        example_product=("product_name", "first"),
    )
    .reset_index()
    .sort_values(["depth", "source_category_code"], na_position="last")
    .reset_index(drop=True)
)

print("실제 최종 매핑 대상 카테고리 수:", len(deepest_summary_df))
display(deepest_summary_df)


## 2. 브랜드 유니크 추출

In [ ]:
brand_rows = []

for item in products:
    brand = item.get("brand") or {}
    product = item.get("product") or {}

    brand_rows.append({
        "source_brand_code": brand.get("brand_code"),
        "name_ko": brand.get("name_ko"),
        "name_en": brand.get("name_en"),
        "country_code": brand.get("nation_code"),
        "country_name": brand.get("nation_name"),
        "since_year": brand.get("since_year"),
        "goods_no": product.get("goods_no"),
        "product_name": product.get("name"),
    })

brand_raw_df = pd.DataFrame(brand_rows)

brand_summary_df = (
    brand_raw_df
    .groupby(
        [
            "source_brand_code",
            "name_ko",
            "name_en",
            "country_code",
            "country_name",
            "since_year",
        ],
        dropna=False,
    )
    .agg(
        product_count=("goods_no", "nunique"),
        example_goods_no=("goods_no", "first"),
        example_product=("product_name", "first"),
    )
    .reset_index()
    .sort_values(
        ["product_count", "name_ko"],
        ascending=[False, True],
        na_position="last",
    )
    .reset_index(drop=True)
)

print("유니크 브랜드 수:", len(brand_summary_df))
display(brand_summary_df)


## 3. Human-in-the-Loop 리뷰 테이블 생성

### 카테고리
- `review_status`: `PENDING`, `MAP`, `IGNORE`
- `feedit_category_code`: FEEDIT 표준 카테고리 코드
- `feedit_category_name`: FEEDIT 표준 카테고리명
- `note`: 사람 검수 메모

### 브랜드
- `review_status`: `PENDING`, `MAP`, `CREATE`, `IGNORE`
- `feedit_brand_id`: 기존 FEEDIT Brand ID가 있으면 입력
- `feedit_brand_name`: FEEDIT 표준 브랜드명
- `note`: 검수 메모

`CREATE`는 FEEDIT Brand에 아직 없는 신규 브랜드로 판단할 때 사용합니다.


In [ ]:
category_review_df = deepest_summary_df.copy()

category_review_df.insert(0, "review_status", "PENDING")
category_review_df["feedit_category_code"] = ""
category_review_df["feedit_category_name"] = ""
category_review_df["note"] = ""

category_review_df = category_review_df[
    [
        "review_status",
        "depth",
        "source_category_code",
        "source_category_name",
        "product_count",
        "example_goods_no",
        "example_product",
        "feedit_category_code",
        "feedit_category_name",
        "note",
    ]
]

brand_review_df = brand_summary_df.copy()

brand_review_df.insert(0, "review_status", "PENDING")
brand_review_df["feedit_brand_id"] = ""
brand_review_df["feedit_brand_name"] = ""
brand_review_df["note"] = ""

brand_review_df = brand_review_df[
    [
        "review_status",
        "source_brand_code",
        "name_ko",
        "name_en",
        "country_code",
        "country_name",
        "since_year",
        "product_count",
        "example_goods_no",
        "example_product",
        "feedit_brand_id",
        "feedit_brand_name",
        "note",
    ]
]

display(category_review_df)
display(brand_review_df)


## 4. 리뷰용 CSV 저장

In [ ]:
CATEGORY_REVIEW_PATH = OUTPUT_DIR / "musinsa_category_mapping_review.csv"
BRAND_REVIEW_PATH = OUTPUT_DIR / "musinsa_brand_mapping_review.csv"

category_review_df.to_csv(
    CATEGORY_REVIEW_PATH,
    index=False,
    encoding="utf-8-sig",
)

brand_review_df.to_csv(
    BRAND_REVIEW_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("저장 완료")
print("-", CATEGORY_REVIEW_PATH.resolve())
print("-", BRAND_REVIEW_PATH.resolve())


## 5. 사람이 CSV를 직접 수정

위 CSV 파일을 Excel/VS Code 등으로 열어서 `review_status`와 `feedit_*` 컬럼을 채우세요.

예시:

### Category
| review_status | source_category_code | source_category_name | feedit_category_code | feedit_category_name |
|---|---|---|---|---|
| MAP | 001010 | 긴소매 티셔츠 | TOP_TSHIRT_LONG | 긴소매 티셔츠 |

### Brand
| review_status | source_brand_code | name_ko | feedit_brand_id | feedit_brand_name |
|---|---|---|---|---|
| MAP | trillion | 트릴리온 | 15 | 트릴리온 |
| CREATE | newbrand | 신규브랜드 |  | 신규브랜드 |

수정한 뒤 다음 셀을 실행합니다.


In [ ]:
category_reviewed_df = pd.read_csv(
    CATEGORY_REVIEW_PATH,
    dtype={
        "source_category_code": "string",
        "feedit_category_code": "string",
    },
    keep_default_na=False,
)

brand_reviewed_df = pd.read_csv(
    BRAND_REVIEW_PATH,
    dtype={
        "source_brand_code": "string",
        "feedit_brand_id": "string",
    },
    keep_default_na=False,
)

print("CATEGORY STATUS")
display(category_reviewed_df["review_status"].value_counts(dropna=False))

print("BRAND STATUS")
display(brand_reviewed_df["review_status"].value_counts(dropna=False))


## 6. 검수 오류 확인

In [ ]:
CATEGORY_ALLOWED = {"PENDING", "MAP", "IGNORE"}
BRAND_ALLOWED = {"PENDING", "MAP", "CREATE", "IGNORE"}

category_status_invalid = category_reviewed_df[
    ~category_reviewed_df["review_status"].isin(CATEGORY_ALLOWED)
]

brand_status_invalid = brand_reviewed_df[
    ~brand_reviewed_df["review_status"].isin(BRAND_ALLOWED)
]

category_map_missing = category_reviewed_df[
    (category_reviewed_df["review_status"] == "MAP")
    & (
        (category_reviewed_df["feedit_category_code"].str.strip() == "")
        | (category_reviewed_df["feedit_category_name"].str.strip() == "")
    )
]

brand_map_missing = brand_reviewed_df[
    (brand_reviewed_df["review_status"] == "MAP")
    & (
        (brand_reviewed_df["feedit_brand_id"].str.strip() == "")
        | (brand_reviewed_df["feedit_brand_name"].str.strip() == "")
    )
]

brand_create_missing = brand_reviewed_df[
    (brand_reviewed_df["review_status"] == "CREATE")
    & (brand_reviewed_df["feedit_brand_name"].str.strip() == "")
]

duplicate_category_source = category_reviewed_df[
    category_reviewed_df.duplicated(
        subset=["source_category_code"],
        keep=False,
    )
].sort_values("source_category_code")

duplicate_brand_source = brand_reviewed_df[
    brand_reviewed_df.duplicated(
        subset=["source_brand_code"],
        keep=False,
    )
].sort_values("source_brand_code")


print("잘못된 Category status:", len(category_status_invalid))
display(category_status_invalid)

print("MAP인데 FEEDIT Category가 비어있는 행:", len(category_map_missing))
display(category_map_missing)

print("중복 source category code:", len(duplicate_category_source))
display(duplicate_category_source)

print("잘못된 Brand status:", len(brand_status_invalid))
display(brand_status_invalid)

print("MAP인데 FEEDIT Brand가 비어있는 행:", len(brand_map_missing))
display(brand_map_missing)

print("CREATE인데 FEEDIT Brand Name이 비어있는 행:", len(brand_create_missing))
display(brand_create_missing)

print("중복 source brand code:", len(duplicate_brand_source))
display(duplicate_brand_source)


## 7. 아직 사람이 검수하지 않은 항목만 보기

In [ ]:
pending_categories = category_reviewed_df[
    category_reviewed_df["review_status"] == "PENDING"
]

pending_brands = brand_reviewed_df[
    brand_reviewed_df["review_status"] == "PENDING"
]

print("남은 Category:", len(pending_categories))
display(pending_categories)

print("남은 Brand:", len(pending_brands))
display(pending_brands)


## 8. 최종 매핑 JSON 생성

`PENDING`을 제외한 검수 완료 결과만 JSON으로 내보냅니다.

이 JSON은 이후 `BrandAlias`, `CategoryAlias` DB 적재 스크립트의 입력으로 사용할 수 있습니다.


In [ ]:
category_final = category_reviewed_df[
    category_reviewed_df["review_status"].isin(["MAP", "IGNORE"])
].copy()

brand_final = brand_reviewed_df[
    brand_reviewed_df["review_status"].isin(["MAP", "CREATE", "IGNORE"])
].copy()


category_mapping_records = category_final.to_dict(orient="records")
brand_mapping_records = brand_final.to_dict(orient="records")


CATEGORY_JSON_PATH = OUTPUT_DIR / "musinsa_category_mapping_final.json"
BRAND_JSON_PATH = OUTPUT_DIR / "musinsa_brand_mapping_final.json"


with CATEGORY_JSON_PATH.open("w", encoding="utf-8") as f:
    json.dump(
        category_mapping_records,
        f,
        ensure_ascii=False,
        indent=2,
    )


with BRAND_JSON_PATH.open("w", encoding="utf-8") as f:
    json.dump(
        brand_mapping_records,
        f,
        ensure_ascii=False,
        indent=2,
    )


print("최종 저장")
print("-", CATEGORY_JSON_PATH.resolve())
print("-", BRAND_JSON_PATH.resolve())

print("\nCATEGORY:", len(category_mapping_records))
print("BRAND   :", len(brand_mapping_records))


## 9. 다음 단계

검수 완료 후 다음 구현:

```text
musinsa_category_mapping_final.json
        ↓
CategoryAlias upsert

musinsa_brand_mapping_final.json
        ↓
Brand / BrandAlias upsert
        ↓
Product / ProductSource / ProductSnapshot 적재
```

이렇게 하면 **자동 매핑이 틀렸을 때 DB를 오염시키지 않고, 사람이 승인한 값만 정규화 DB에 반영**할 수 있습니다.
